## Ablation study

Reviewing results in ablation study:

1- R2: normal and anomaly state (just 2 states)
We just claim 2 states: normal, anomal (identify by the time series anomaly detection IQR on weekly issue_closed)
2- R3: process categories 
5 category based on the regex on CONVENTIONAL_TYPES (link)
3- the combination of states and categories

Baseline: 
whole event log without any partitioning

Setups:
3 miners
DISCOVERY_MINERS = [
    "inductive_IMf_noise_0.2",
    "split_miner_eps_0.2_eta_0.5",
    "heuristics_dep_0.5",
]
5 repository (to be unbiased about the quality of commit messages)
3 of them claimed that considering the conventional commit
https://github.com/commitizen-tools/commitizen
https://github.com/yargs/yargs
https://github.com/semantic-release/semantic-release


2 of the are famous finance repository, they are popular in terms of stars and forks
https://github.com/tauricresearch/tradingagents
https://github.com/HKUDS/Vibe-Trading


Descriptive metrics: like cycle time,..
Quality metrics: Precision, fitness and f score (token replay)
Model complexity metrics: size, cfc, simplicity


reporting f-score for whole log
reporitng weighted average :
normal/anomaly, commit categories, anomaly categories — sum(f_score * n_cases) / sum(n_cases) on that sheet, excluding whole



In [1]:
from pathlib import Path
import pandas as pd

DATASETS = [
    "commitizen",
    "TradingAgents",
    "Vibe-Trading",
    "yargs",
    "semantic-release",
]
MINERS = [
    "inductive_IMf_noise_0.2",
    "split_miner_eps_0.2_eta_0.5",
    "heuristics_dep_0.5",
]
PARTITIONS = [
    ("whole", "commit_categories", True),
    ("normal/anomaly states", "anomaly_normal", False),
    ("commit categories", "commit_categories", False),
    ("states + categories", "anomaly_categories", False),
]
TABLES = Path("results/tables")


def weighted_f(df, miner, whole_only):
    sub = df[df["discovery_method"] == miner]
    if whole_only:
        return sub.loc[sub["log_name"] == "whole", "f_score"].iloc[0]
    sub = sub[sub["log_name"] != "whole"]
    return (sub["f_score"] * sub["n_cases"]).sum() / sub["n_cases"].sum()


rows = []
for miner in MINERS:
    for label, sheet, whole_only in PARTITIONS:
        row = {"discovery_method": miner, "partition": label}
        for dataset in DATASETS:
            df = pd.read_excel(
                TABLES / dataset / "consolidated_results_tables.xlsx",
                sheet_name=sheet,
            )
            row[dataset] = weighted_f(df, miner, whole_only)
        rows.append(row)

comparison = pd.DataFrame(rows).set_index(["discovery_method", "partition"])

out_path = TABLES / "compare" / "f_score_ablation_comparison.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
comparison.to_csv(out_path)
print(f"Saved {out_path}")
comparison


Saved results\tables\compare\f_score_ablation_comparison.csv


commitizen  TradingAgents  \
discovery_method            partition                                          
inductive_IMf_noise_0.2     whole                    0.612254       0.679391   
                            normal/anomaly states    0.532808       0.708584   
                            commit categories        0.524767       0.618372   
                            states + categories      0.484548       0.661372   
split_miner_eps_0.2_eta_0.5 whole                    0.531106       0.387004   
                            normal/anomaly states    0.532836       0.418569   
                            commit categories        0.550592       0.387566   
                            states + categories      0.593382       0.458153   
heuristics_dep_0.5          whole                    0.517981       0.379064   
                            normal/anomaly states    0.519558       0.493185   
                            commit categories        0.542605       0.466700   
                            states + categories      0.553092       0.489230   

                                                   Vibe-Trading     yargs  \
discovery_method            partition                                       
inductive_IMf_noise_0.2     whole                      0.306520  0.466746   
                            normal/anomaly states      0.392996  0.513422   
                            commit categories          0.627709  0.475898   
                            states + categories        0.494225  0.502697   
split_miner_eps_0.2_eta_0.5 whole                      0.575421  0.637713   
                            normal/anomaly states      0.656604  0.555065   
                            commit categories          0.656049  0.651091   
                            states + categories        0.656622  0.693177   
heuristics_dep_0.5          whole                      0.388835  0.442786   
                            normal/anomaly states      0.555543  0.455536   
                            commit categories          0.556779  0.536600   
                            states + categories        0.625139  0.504920   

                                                   semantic-release  
discovery_method            partition                                
inductive_IMf_noise_0.2     whole                          0.276562  
                            normal/anomaly states          0.262491  
                            commit categories              0.333440  
                            states + categories            0.284136  
split_miner_eps_0.2_eta_0.5 whole                          0.401086  
                            normal/anomaly states          0.474362  
                            commit categories              0.518463  
                            states + categories            0.610735  
heuristics_dep_0.5          whole                          0.513899  
                            normal/anomaly states          0.544794  
                            commit categories              0.599962  
                            states + categories            0.599260